In [ ]:
!pip install bert-score sentence-transformers

In [ ]:
import json, re, gc, torch
from transformers import (BartTokenizerFast, BartForConditionalGeneration,
                           Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq)
from datasets import Dataset
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from bert_score import score as bertscore
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
import json, re, gc, torch, os
from transformers import (BartTokenizerFast, BartForConditionalGeneration,
                           Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq)
from datasets import Dataset
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from bert_score import score as bertscore
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def load_processed(split):
    # Replace 'your-dataset-folder-name' with the exact name of the dataset folder in your right sidebar
    dataset_dir = "/kaggle/input/datasets/rm2110/data-json"
    
    file_path = os.path.join(dataset_dir, f"processed_{split}.json")
    
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_processed("train")
val_data = load_processed("val")
test_data = load_processed("test")

sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

BACKGROUND_CUES = ["problem","challenge","existing","prior work","however","limitation","propose","address","motivated","prior","previous","traditionally","remains","lack of","difficult","issue"]
METHOD_CUES = ["method","approach","algorithm","model","framework","technique","using","based on","we present","we introduce","we develop","architecture","training","trained","design"]
RESULT_CUES = ["show","results","achieve","outperform","accuracy","evaluate","experiment","demonstrate","improve","performance","state-of-the-art","compared to","significantly","we find"]

def classify_sentence(s):
    s = s.lower()
    sc = {"background": sum(1 for c in BACKGROUND_CUES if c in s),
          "method": sum(1 for c in METHOD_CUES if c in s),
          "result": sum(1 for c in RESULT_CUES if c in s)}
    best = max(sc, key=sc.get)
    return "background" if sc[best] == 0 else best

def mmr_filter_sentences(cited_abstracts, target_abstract, max_sentences=10, lambda_param=0.7):
    all_s = []
    for a in cited_abstracts:
        all_s.extend(sent_tokenize(a))
    if len(all_s) <= max_sentences:
        return all_s
    q = sbert_model.encode(target_abstract, convert_to_tensor=True)
    e = sbert_model.encode(all_s, convert_to_tensor=True)
    rel = util.cos_sim(e, q).squeeze()
    sel, rem = [], list(range(len(all_s)))
    while len(sel) < max_sentences and rem:
        if not sel:
            best = max(rem, key=lambda i: rel[i].item())
        else:
            se = e[sel]
            best = max(rem, key=lambda i: lambda_param*rel[i].item() - (1-lambda_param)*util.cos_sim(e[i], se).max().item())
        sel.append(best); rem.remove(best)
    sel.sort()
    return [all_s[i] for i in sel]

def format_input_v2(example, max_sentences=10):
    filtered = mmr_filter_sentences(example["cited_abstracts"], example["target_abstract"], max_sentences)
    g = {"background": [], "method": [], "result": []}
    for s in filtered:
        g[classify_sentence(s)].append(s)
    parts = [f"<target> {example['target_abstract']}"]
    for role in ["background", "method", "result"]:
        if g[role]:
            parts.append(f"<{role}> " + " ".join(g[role]))
    return " ".join(parts)

MAX_INPUT_LEN = 768
MAX_TARGET_LEN = 128

print("Everything reloaded and ready.")

In [ ]:
model_large = BartForConditionalGeneration.from_pretrained("facebook/bart-large", use_safetensors=True)
tokenizer_large = BartTokenizerFast.from_pretrained("facebook/bart-large")

special_tokens = {"additional_special_tokens": ["<target>", "<background>", "<method>", "<result>"]}
tokenizer_large.add_special_tokens(special_tokens)
model_large.resize_token_embeddings(len(tokenizer_large))

embedding_weights = model_large.get_input_embeddings().weight
print("Any NaN in embeddings right after resize?", torch.isnan(embedding_weights).any().item())
print("Min/Max embedding values:", embedding_weights.min().item(), embedding_weights.max().item())

new_token_embeds = embedding_weights[-4:]
print("New token embeddings:\n", new_token_embeds)

In [ ]:
model_large = BartForConditionalGeneration.from_pretrained(
    "facebook/bart-large",
    use_safetensors=True,
    torch_dtype=torch.float32   # force full precision, overriding whatever the checkpoint defaults to
)
tokenizer_large = BartTokenizerFast.from_pretrained("facebook/bart-large")

special_tokens = {"additional_special_tokens": ["<target>", "<background>", "<method>", "<result>"]}
tokenizer_large.add_special_tokens(special_tokens)
model_large.resize_token_embeddings(len(tokenizer_large))

embedding_weights = model_large.get_input_embeddings().weight
print("Dtype now:", embedding_weights.dtype)
print("Any NaN?", torch.isnan(embedding_weights).any().item())

In [ ]:
# Rebuild dataset
train_subset = train_data[:20000]
val_subset = val_data[:1000]

def build_dataset(data):
    return Dataset.from_dict({
        "input_text": [format_input_v2(ex) for ex in data],
        "target_text": [ex["gold_summary"] for ex in data],
    })

train_ds = build_dataset(train_subset)
val_ds = build_dataset(val_subset)

def tokenize_fn(batch):
    model_inputs = tokenizer_large(batch["input_text"], max_length=MAX_INPUT_LEN, truncation=True, padding="max_length")
    labels = tokenizer_large(batch["target_text"], max_length=MAX_TARGET_LEN, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
val_tokenized = val_ds.map(tokenize_fn, batched=True, remove_columns=val_ds.column_names)

print("Dataset rebuilt:", train_tokenized)

In [ ]:
# Move to GPU
model_large.to(device)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_large_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    max_grad_norm=1.0,
    save_total_limit=1,
    predict_with_generate=True,
    fp16=False,
    logging_steps=50,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model_large, args=training_args,
    train_dataset=train_tokenized, eval_dataset=val_tokenized,
    processing_class=tokenizer_large,
    data_collator=DataCollatorForSeq2Seq(tokenizer_large, model=model_large),
)

trainer.train()

In [ ]:
model_large.save_pretrained("./saved_models/bart_large_final")
tokenizer_large.save_pretrained("./saved_models/bart_large_final")
print("Saved.")

In [ ]:
def generate_summary_large(input_text, max_length=128):
    inputs = tokenizer_large(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)
    output_ids = model_large.generate(
        **inputs, max_length=max_length, num_beams=4,
        no_repeat_ngram_size=3, length_penalty=1.2, early_stopping=True,
    )
    return tokenizer_large.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
for i in range(3):
    example = val_subset[i]
    generated = generate_summary_large(format_input_v2(example))
    print(f"--- Example {i} ---")
    print("GOLD:\n", example["gold_summary"])
    print("\nGENERATED (bart-large):\n", generated)
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)    print("\n" + "="*80 + "\n")

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

test_subset_300 = test_data[:300]
generated_large, gold_large = [], []

for ex in test_subset_300:
    generated_large.append(generate_summary_large(format_input_v2(ex)))
    gold_large.append(ex["gold_summary"])

r1 = [scorer.score(g, gen)["rouge1"].fmeasure for gen, g in zip(generated_large, gold_large)]
r2 = [scorer.score(g, gen)["rouge2"].fmeasure for gen, g in zip(generated_large, gold_large)]
rL = [scorer.score(g, gen)["rougeL"].fmeasure for gen, g in zip(generated_large, gold_large)]

print("bart-large ROUGE-1:", sum(r1)/len(r1))
print("bart-large ROUGE-2:", sum(r2)/len(r2))
print("bart-large ROUGE-L:", sum(rL)/len(rL))

P, R, F1 = bertscore(generated_large, gold_large, lang="en", verbose=True, batch_size=16)
print("bart-large BERTScore F1:", F1.mean().item())

In [ ]:
P, R, F1 = bertscore(generated_large, gold_large, lang="en", verbose=True, batch_size=16)
print("bart-large BERTScore F1:", F1.mean().item())

In [ ]:
final_model_comparison = {
    "v2_bart_base_3k": {"rouge1": 0.2970, "rouge2": 0.0582, "rougeL": 0.1689, "bertscore_f1": 0.8515},
    "bart_large_20k": {"rouge1": 0.32121717756578094, "rouge2": 0.06630442048089903, "rougeL": 0.17703721762808539, "bertscore_f1": 0.8495392203330994},
    "final_choice": "bart-large (20K)"
}
with open("final_model_comparison.json", "w") as f:
    json.dump(final_model_comparison, f, indent=2)
print("Saved.")

In [ ]:
import json, re, gc, torch
from transformers import BartTokenizerFast, BartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from bert_score import score as bertscore
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

device = "cuda" if torch.cuda.is_available() else "cpu"

def load_processed(split):
    # Replace 'your-dataset-folder-name' with the exact name of the dataset folder in your right sidebar
    dataset_dir = "/kaggle/input/datasets/rm2110/data-json"
    
    file_path = os.path.join(dataset_dir, f"processed_{split}.json")
    
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_processed("train")
val_data = load_processed("val")
test_data = load_processed("test")

sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

BACKGROUND_CUES = ["problem","challenge","existing","prior work","however","limitation","propose","address","motivated","prior","previous","traditionally","remains","lack of","difficult","issue"]
METHOD_CUES = ["method","approach","algorithm","model","framework","technique","using","based on","we present","we introduce","we develop","architecture","training","trained","design"]
RESULT_CUES = ["show","results","achieve","outperform","accuracy","evaluate","experiment","demonstrate","improve","performance","state-of-the-art","compared to","significantly","we find"]

def classify_sentence(s):
    s = s.lower()
    sc = {"background": sum(1 for c in BACKGROUND_CUES if c in s),
          "method": sum(1 for c in METHOD_CUES if c in s),
          "result": sum(1 for c in RESULT_CUES if c in s)}
    best = max(sc, key=sc.get)
    return "background" if sc[best] == 0 else best

def mmr_filter_sentences(cited_abstracts, target_abstract, max_sentences=10, lambda_param=0.7):
    all_s = []
    for a in cited_abstracts:
        all_s.extend(sent_tokenize(a))
    if len(all_s) <= max_sentences:
        return all_s
    q = sbert_model.encode(target_abstract, convert_to_tensor=True)
    e = sbert_model.encode(all_s, convert_to_tensor=True)
    rel = util.cos_sim(e, q).squeeze()
    sel, rem = [], list(range(len(all_s)))
    while len(sel) < max_sentences and rem:
        if not sel:
            best = max(rem, key=lambda i: rel[i].item())
        else:
            se = e[sel]
            best = max(rem, key=lambda i: lambda_param*rel[i].item() - (1-lambda_param)*util.cos_sim(e[i], se).max().item())
        sel.append(best); rem.remove(best)
    sel.sort()
    return [all_s[i] for i in sel]

def format_input_v2(example, max_sentences=10):
    filtered = mmr_filter_sentences(example["cited_abstracts"], example["target_abstract"], max_sentences)
    g = {"background": [], "method": [], "result": []}
    for s in filtered:
        g[classify_sentence(s)].append(s)
    parts = [f"<target> {example['target_abstract']}"]
    for role in ["background", "method", "result"]:
        if g[role]:
            parts.append(f"<{role}> " + " ".join(g[role]))
    return " ".join(parts)

print("Everything reloaded and ready.")

In [ ]:
import torch
from transformers import BartTokenizerFast, BartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer_large2 = BartTokenizerFast.from_pretrained("facebook/bart-large")
model_large2 = BartForConditionalGeneration.from_pretrained(
    "facebook/bart-large", use_safetensors=True, torch_dtype=torch.float32
)

special_tokens = {"additional_special_tokens": ["<target>", "<background>", "<method>", "<result>"]}
tokenizer_large2.add_special_tokens(special_tokens)
model_large2.resize_token_embeddings(len(tokenizer_large2))
model_large2.gradient_checkpointing_enable()
model_large2.enable_input_require_grads()
model_large2.to(device)

print("Dtype:", model_large2.get_input_embeddings().weight.dtype)
print("GPU allocated:", torch.cuda.memory_allocated()/1e9, "GB")

train_subset = train_data[:20000]
val_subset = val_data[:1000]

def build_dataset(data):
    return Dataset.from_dict({
        "input_text": [format_input_v2(ex) for ex in data],
        "target_text": [ex["gold_summary"] for ex in data],
    })

train_ds = build_dataset(train_subset)
val_ds = build_dataset(val_subset)

MAX_INPUT_LEN_C = 768

def tokenize_fn(batch):
    model_inputs = tokenizer_large2(batch["input_text"], max_length=MAX_INPUT_LEN_C, truncation=True, padding="max_length")
    labels = tokenizer_large2(batch["target_text"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
val_tokenized = val_ds.map(tokenize_fn, batched=True, remove_columns=val_ds.column_names)

print("Dataset ready:", train_tokenized)

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_large_lr3e5",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    max_grad_norm=1.0,
    save_total_limit=1,
    predict_with_generate=True,
    fp16=False,
    logging_steps=50,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model_large2, args=training_args,
    train_dataset=train_tokenized, eval_dataset=val_tokenized,
    processing_class=tokenizer_large2,
    data_collator=DataCollatorForSeq2Seq(tokenizer_large2, model=model_large2),
)

trainer.train()

# ✅ Save immediately after training completes — don't skip this
model_large2.save_pretrained("./saved_models/bart_large_lr3e5_final")
tokenizer_large2.save_pretrained("./saved_models/bart_large_lr3e5_final")
print("✅ MODEL SAVED SUCCESSFULLY to ./saved_models/bart_large_lr3e5_final")

In [ ]:
def generate_summary_large2(input_text, max_length=128):
    inputs = tokenizer_large2(input_text, return_tensors="pt", max_length=MAX_INPUT_LEN_C, truncation=True).to(device)
    output_ids = model_large2.generate(
        **inputs, max_length=max_length, num_beams=4,
        no_repeat_ngram_size=3, length_penalty=1.2, early_stopping=True,
    )
    return tokenizer_large2.decode(output_ids[0], skip_special_tokens=True)

test_subset_300 = test_data[:300]
generated_lr3, gold_lr3 = [], []

for ex in test_subset_300:
    generated_lr3.append(generate_summary_large2(format_input_v2(ex)))
    gold_lr3.append(ex["gold_summary"])

r1 = [scorer.score(g, gen)["rouge1"].fmeasure for gen, g in zip(generated_lr3, gold_lr3)]
r2 = [scorer.score(g, gen)["rouge2"].fmeasure for gen, g in zip(generated_lr3, gold_lr3)]
rL = [scorer.score(g, gen)["rougeL"].fmeasure for gen, g in zip(generated_lr3, gold_lr3)]

print("bart-large (lr=3e-5) ROUGE-1:", sum(r1)/len(r1))
print("bart-large (lr=3e-5) ROUGE-2:", sum(r2)/len(r2))
print("bart-large (lr=3e-5) ROUGE-L:", sum(rL)/len(rL))

P, R, F1 = bertscore(generated_lr3, gold_lr3, lang="en", verbose=True, batch_size=16)
print("bart-large (lr=3e-5) BERTScore F1:", F1.mean().item())

In [ ]:
final_results = {
    "model": "bart-large (20K train, lr=3e-5, fp32, AdamW)",
    "rouge1": 0.32694826290108514,
    "rouge2": 0.06999107535861601,
    "rougeL": 0.1815618256531125,
    "bertscore_f1": 0.8506642580032349,
}

with open("FINAL_MODEL_RESULTS.json", "w") as f:
    json.dump(final_results, f, indent=2)

with open("FINAL_generated_summaries.json", "w") as f:
    json.dump({"generated": generated_lr3, "gold": gold_lr3}, f, indent=2)

print("Final results saved.")

In [ ]:
import os
for root, dirs, files in os.walk("./saved_models"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import shutil
shutil.make_archive("bart_large_final_model", "zip", "./saved_models/bart_large_lr3e5_final")
shutil.make_archive("all_results", "zip", ".", "FINAL_MODEL_RESULTS.json")

print("Zipped and ready to download.")